In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error

# 1) Load data
df = pd.read_csv("../Capstone/Capstone Dataset (New).csv")

X = df.drop(
    ["Array type", "Predicted angular insertion depth (deg)",
     "Actual angular insertion depth (deg)"],
    axis=1
)
y = df["Actual angular insertion depth (deg)"]

# 2) Define model and search space
base_tree = DecisionTreeRegressor(random_state=42)
ada = AdaBoostRegressor(
    estimator=base_tree,    
    random_state=42
)

param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "estimator__max_depth": [1, 2, 3, 4],
    "estimator__min_samples_leaf": [1, 2, 5],
}

# 3) Grid search
gs = GridSearchCV(
    estimator=ada,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=5,       # change to LeaveOneOut() if dataset is very small
    n_jobs=-1,
    verbose=1,
    refit=True
)
gs.fit(X, y)

print("Best params:", gs.best_params_)
print("CV best MAE:", -gs.best_score_)

Fitting 5 folds for each of 192 candidates, totalling 960 fits


C:\Users\shell\anaconda3\Lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Best params: {'estimator__max_depth': 4, 'estimator__min_samples_leaf': 5, 'learning_rate': 0.1, 'n_estimators': 500}
CV best MAE: 43.42246436920358


In [6]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import LeaveOneOut

#Loading dataset
df = pd.read_csv("Capstone Dataset (New).csv")

#Separate=ing features (X) and target (y)
X = df.drop(
    ["Array type", "Predicted angular insertion depth (deg)",
     "Actual angular insertion depth (deg)"],
    axis=1
)
y = df["Actual angular insertion depth (deg)"]

#Leave-One-Out Method
loo = LeaveOneOut()

y_true_all = []
y_pred_all = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

#AdaBoost regressor (Using best parameters)
    ada = AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=4, random_state=42),
        n_estimators=500,
        learning_rate=0.1,
        random_state=42,
    )

#Fit model
    ada.fit(X_train, y_train)

#Predict the held-out sample
    y_pred = ada.predict(X_test)[0]

    # Collect predictions and truth
    y_pred_all.append(y_pred)
    y_true_all.append(y_test.values[0])

# Evaluation (overall MAE across all LOO folds)
mae = mean_absolute_error(y_true_all, y_pred_all)
print("MAE:", mae)

MAE: 36.761919034144235
